# Explore PubMed baseline files

Downloads a few PubMed baseline files and shows, for each one: how many records pass the pipeline's filters,
which publication years it covers, and some sample (title, abstract) pairs. Use it to choose `--n-files` /
`--file-start` before running `colab.ipynb`. Parsing uses the same code as the pipeline (`tsdae_sbert_pubmed.prepare_pubmed`).

On Colab, the first cell clones this repo into `/content` (no Drive needed). Run locally from the repo root or from `notebook/`.

In [ ]:
import os

try:
    import google.colab  # noqa: F401  (only to detect Colab)
    REPO_DIR = '/content/tsdae-sbert-pubmed'
    if not os.path.isdir(REPO_DIR):
        !git clone -q https://github.com/mohpezeshki/tsdae-sbert-pubmed.git {REPO_DIR}
    RAW_DIR = '/content/pubmed_raw'
    %cd {REPO_DIR}
    !pip -q install -r requirements.txt
except ImportError:  # local run
    if os.path.basename(os.getcwd()) == 'notebook':
        os.chdir('..')
    RAW_DIR = 'data/pubmed_raw'

In [ ]:
import gzip
import re
from collections import Counter
from pathlib import Path

from tsdae_sbert_pubmed.config import Config
from tsdae_sbert_pubmed.prepare_pubmed import download_file, iter_articles, list_baseline_files

cfg = Config()  # the pipeline's defaults, for the length filters
names = list_baseline_files()
print(len(names), 'files on server:', names[0], '...', names[-1])

Pick files by index into the sorted list (negative = from the end). File order is **not** chronological.

In [ ]:
FILE_INDICES = [0, -1]  # e.g. first and newest; add more to compare eras
paths = [download_file(names[i], Path(RAW_DIR)) for i in FILE_INDICES]

In [ ]:
YEAR_RE = re.compile(rb'<PubDate>\s*<Year>(\d{4})</Year>')

for path in paths:
    with gzip.open(path, 'rb') as f:
        years = Counter(int(y) for y in YEAR_RE.findall(f.read()))
    pairs = list(iter_articles(path))
    kept = [(t, a) for t, a in pairs
            if len(t.split()) >= cfg.min_title_words and len(a.split()) >= cfg.min_abstract_words]
    print(f'{path.name}: {sum(years.values())} dated records, {len(pairs)} English title+abstract, '
          f'{len(kept)} pass length filters')
    print('  most common years:', years.most_common(3))

Sample pairs from the last file (what TSDAE sentences and dev queries look like):

In [ ]:
for title, abstract in kept[:3]:
    print(title, '\n ', abstract[:400], '...\n')